In [ ]:
# ════════════════════════════════════════════════════════════
# TTS — paragraph -> speech audio  (edge-tts: Microsoft Neural TTS, free)
# Writes an .mp3 and prints its duration (feed that in as S in Milestone 2).
# ════════════════════════════════════════════════════════════
import os, sys, json, asyncio, subprocess
from pathlib import Path

# ── EDIT THESE ───────────────────────────────────────────────
PARAGRAPH = (
    "Hey there, it's really good to see you. So, here's the deal, I'm here to make "
    "your day a little easier, whether that's answering a quick question or walking "
    "you through something step by step. There's honestly no rush at all, we'll just "
    "take it at whatever pace feels right for you. So go ahead, get comfortable, and "
    "whenever you're ready, just say the word and we'll jump right in."
)

# Most natural / human-sounding voices (newest "conversational" set) — pick one:
#   en-US-AvaMultilingualNeural     female, warm & expressive  (default)
#   en-US-AndrewMultilingualNeural  male,   relaxed & friendly
#   en-US-EmmaMultilingualNeural    female, bright & casual
#   en-US-BrianMultilingualNeural   male,   easy-going
#   en-GB-SoniaNeural / en-AU-NatashaNeural  British / Australian
# (The older en-US-JennyNeural / AriaNeural / GuyNeural sound more robotic.)
VOICE       = "en-US-AvaMultilingualNeural"
OUTPUT_PATH = "./assets/paragraph_tts.mp3"

# Gentle delivery tuning (leave at defaults for natural; tweak to taste):
RATE   = "-4%"     # slightly slower reads warmer/less rushed  (e.g. "-10%", "+0%")
PITCH  = "+0Hz"    # try "-2Hz" for a calmer tone
VOLUME = "+0%"
# ─────────────────────────────────────────────────────────────

# Make sure the deps are present (works on a fresh machine / Colab).
for pkg, mod in [("edge-tts", "edge_tts"), ("nest_asyncio", "nest_asyncio")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import edge_tts, nest_asyncio
nest_asyncio.apply()   # lets asyncio run inside the notebook's event loop

os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

async def _generate():
    await edge_tts.Communicate(PARAGRAPH, VOICE,
                               rate=RATE, pitch=PITCH, volume=VOLUME).save(OUTPUT_PATH)
asyncio.get_event_loop().run_until_complete(_generate())

if not os.path.exists(OUTPUT_PATH) or os.path.getsize(OUTPUT_PATH) == 0:
    raise RuntimeError("edge-tts produced no audio — check the voice id and your internet connection.")

# Duration via ffprobe (optional — needs ffmpeg on PATH).
def _duration(p):
    try:
        out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                              "-of", "json", p], capture_output=True, text=True).stdout
        return float(json.loads(out)["format"]["duration"])
    except Exception:
        return None

dur = _duration(OUTPUT_PATH)
print(f"done -> {OUTPUT_PATH}")
print(f"  voice    : {VOICE}  (rate {RATE}, pitch {PITCH})")
print(f"  words    : {len(PARAGRAPH.split())}")
print(f"  size     : {os.path.getsize(OUTPUT_PATH) // 1024} KB")
print(f"  duration : {dur:.2f}s   <- use this as S in Milestone 2" if dur
      else "  duration : (install ffmpeg to measure)")

# Tip: to hear every available voice ->  run in a terminal:  edge-tts --list-voices


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Attach the TTS audio onto a video at the speaking window
# Places the audio from t = OFFSET_SEC (3.0s) onward — i.e. exactly over the
# speaking segment of an idle(3) + speaking(S) + idle(3) sequenced clip.
# The video is copied as-is (no re-encode); before 3s and after the audio,
# the track stays silent.
# (Self-contained. Needs ffmpeg + ffprobe on PATH. Run the TTS cell first.)
# ════════════════════════════════════════════════════════════
import os, json, subprocess

# ── EDIT THESE ───────────────────────────────────────────────
VIDEO_PATH   = './assets/person_03_sequenced.mp4'    # idle+speaking+idle clip (6+S long)
AUDIO_PATH   = './assets/paragraph_tts.mp3'          # TTS audio from the cell above
MUXED_OUTPUT = './assets/person_03_with_audio.mp4'
OFFSET_SEC   = 3.0    # audio starts here (= the idle lead-in length)
# ─────────────────────────────────────────────────────────────

def _dur(p):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                          "-of", "json", str(p)], capture_output=True, text=True).stdout
    try:
        return float(json.loads(out)["format"]["duration"])
    except Exception:
        return None

for f in (VIDEO_PATH, AUDIO_PATH):
    if not os.path.exists(f):
        raise FileNotFoundError(f"Not found: {f}")
os.makedirs(os.path.dirname(MUXED_OUTPUT) or ".", exist_ok=True)

vdur, adur = _dur(VIDEO_PATH), _dur(AUDIO_PATH)
delay_ms = int(round(OFFSET_SEC * 1000))

# Pad the audio with OFFSET_SEC of leading silence, then mux it over the video.
# Video is stream-copied; output length follows the (longer) video, so the
# trailing idle stays silent.
cmd = [
    "ffmpeg", "-y", "-i", VIDEO_PATH, "-i", AUDIO_PATH,
    "-filter_complex", f"[1:a]adelay=delays={delay_ms}:all=1[a]",
    "-map", "0:v", "-map", "[a]",
    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
    MUXED_OUTPUT,
]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("ffmpeg mux failed:\n" + r.stderr[-1500:])

out_dur = _dur(MUXED_OUTPUT)
print(f"done -> {MUXED_OUTPUT}")
print(f"  video : {vdur:.2f}s" if vdur else "  video : ?")
print(f"  audio : {adur:.2f}s  placed at {OFFSET_SEC:.1f}s -> {OFFSET_SEC + (adur or 0):.2f}s")
print(f"  total : {out_dur:.2f}s (silent outside the speaking window)")
if vdur and adur:
    spk = vdur - 2 * OFFSET_SEC          # the video's speaking window S
    if OFFSET_SEC + adur > vdur + 0.05:
        print(f"  NOTE: audio runs {OFFSET_SEC + adur - vdur:.2f}s past the video end and will be cut.")
        print(f"        Rebuild the video with S = {adur:.2f}s so the speaking window fits the audio.")
    else:
        fit = 'fits' if adur <= spk + 0.05 else 'slightly over the window'
        print(f"  speaking window S = {spk:.2f}s ; audio = {adur:.2f}s  ({fit})")
